In [7]:
from google.colab import drive
drive.mount('/content/drive')

# cd to the *root* of career-ml, NOT to scripts
%cd "/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml


In [8]:
import json
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


In [9]:
ROLE_PROFILE_PATH = "skill_gap/role_skill_profiles.csv"
CAREER_LADDER_PATH = "career_path/career_ladders.json"

role_profiles = pd.read_csv(ROLE_PROFILE_PATH)

with open(CAREER_LADDER_PATH, "r") as f:
    CAREER_LADDERS = json.load(f)

role_profiles.head()


,role_id,skill_id,frequency,importance
0,AI_ML_ENGINEER_INT,SK001,16,0.020627
1,AI_ML_ENGINEER_INT,SK002,3,0.003868
2,AI_ML_ENGINEER_INT,SK003,21,0.027073
3,AI_ML_ENGINEER_INT,SK004,73,0.146752
4,AI_ML_ENGINEER_INT,SK006,2,0.007306


In [10]:
role_skill_matrix = role_profiles.pivot_table(
    index="role_id",
    columns="skill_id",
    values="importance",
    fill_value=0
)


In [11]:
def build_user_vector(user_skill_ids, skill_columns):
    vector = np.zeros(len(skill_columns))
    skill_index = {s: i for i, s in enumerate(skill_columns)}

    for skill_id in user_skill_ids:
        if skill_id in skill_index:
            vector[skill_index[skill_id]] = 1.0

    return vector.reshape(1, -1)


In [12]:
def recommend_careers(user_skill_ids, top_n=5):
    user_vector = build_user_vector(
        user_skill_ids,
        role_skill_matrix.columns
    )

    similarities = cosine_similarity(
        user_vector,
        role_skill_matrix.values
    )[0]

    ranked = pd.DataFrame({
        "role_id": role_skill_matrix.index,
        "match_score": similarities
    }).sort_values("match_score", ascending=False)

    return ranked.head(top_n)


In [13]:
def get_best_career(user_skill_ids):
    top = recommend_careers(user_skill_ids, top_n=1)
    return top.iloc[0]["role_id"], float(top.iloc[0]["match_score"])


In [14]:
def get_next_role(domain, current_role):
    ladder = CAREER_LADDERS.get(domain, [])

    if current_role not in ladder:
        return None

    idx = ladder.index(current_role)

    if idx + 1 < len(ladder):
        return ladder[idx + 1]

    return None


In [15]:
def detect_skill_gap(user_skill_ids, target_role_id, importance_threshold=0.02):
    role_df = role_profiles[role_profiles["role_id"] == target_role_id]

    if role_df.empty:
        raise ValueError(f"No role profile found for role_id={target_role_id}")

    required = role_df[role_df["importance"] >= importance_threshold]

    required_skills = set(required["skill_id"])
    matched = sorted(required_skills & user_skill_ids)
    missing = sorted(required_skills - user_skill_ids)

    readiness = (
        len(matched) / len(required_skills)
        if required_skills else 0.0
    )

    return {
        "readiness_score": round(readiness, 3),
        "matched_skills": matched,
        "missing_skills": missing
    }


In [16]:
def detect_skill_gap(user_skill_ids, target_role_id, importance_threshold=0.02):
    role_df = role_profiles[role_profiles["role_id"] == target_role_id]

    if role_df.empty:
        raise ValueError(f"No role profile found for role_id={target_role_id}")

    required = role_df[role_df["importance"] >= importance_threshold]

    required_skills = set(required["skill_id"])
    matched = sorted(required_skills & user_skill_ids)
    missing = sorted(required_skills - user_skill_ids)

    readiness = (
        len(matched) / len(required_skills)
        if required_skills else 0.0
    )

    return {
        "readiness_score": round(readiness, 3),
        "matched_skills": matched,
        "missing_skills": missing
    }


In [17]:
def simulate_career_path(
    user_skill_ids: set,
    domain: str,
    importance_threshold: float = 0.02
):
    # 1️⃣ Recommend best career
    best_role, confidence = get_best_career(user_skill_ids)

    # 2️⃣ Determine next role in ladder
    next_role = get_next_role(domain, best_role)

    # 3️⃣ If no next role → already at top
    if next_role is None:
        return {
            "domain": domain,
            "recommended_role": best_role,
            "confidence": confidence,
            "message": "You are already at the highest role in this career path."
        }

    # 4️⃣ Skill gap for next role
    gap = detect_skill_gap(
        user_skill_ids=user_skill_ids,
        target_role_id=next_role,
        importance_threshold=importance_threshold
    )

    return {
        "domain": domain,
        "recommended_role": best_role,
        "recommendation_confidence": confidence,
        "next_role": next_role,
        "readiness_score": gap["readiness_score"],
        "matched_skills": gap["matched_skills"],
        "missing_skills": gap["missing_skills"]
    }


In [18]:
user_skill_ids = {
    "SK004",  # python
    "SK031",  # machine learning
    "SK102"   # sql
}

simulate_career_path(
    user_skill_ids=user_skill_ids,
    domain="AI_ML"
)


{'domain': 'AI_ML',
 'recommended_role': 'AI_ML_ENGINEER_INT',
 'confidence': 0.579371902262629,
 'message': 'You are already at the highest role in this career path.'}